# MLOps Assignment 2: LLMOps — API Cost, Latency, and Prompt Version Management

**Course:** MLOps | **Student:** Kush GUnendrasing Munot | **ID:** 2025ae05915 | **Date:** 2026-11-03

**Scenario:** EduAssist AI academic question-answering assistant. Three prompt versions are tested on
the same five questions, their latency, token usage, cost and JSON validity are tracked in MLflow,
and the best version is promoted with the MLflow alias `production`.

**Setup:** Model `gemini-3.1-flash-lite` (Google Gemini API v1beta, native `models.generateContent` REST endpoint),
standard pricing $0.25 / $1.50 per 1M input / output tokens (USD, pricing date 2026-11-03),
MLflow 3.17.2 server running in Podman, Python 3.12.

**Contents:** 1 Setup · 2 Task 1 Latency · 3 Task 2 Tokens and cost · 4 Task 3 Prompt registry,
evaluation and promotion · 5 Final checks

In [1]:
import os
import sys
import json
import time
import platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import mlflow
import jsonschema
from dotenv import load_dotenv

print("Python     :", sys.version.split()[0], "on", platform.platform())
print("requests   :", requests.__version__)
print("mlflow     :", mlflow.__version__)
print("pandas     :", pd.__version__)
print("jsonschema :", jsonschema.__version__)

Python     : 3.12.13 on macOS-26.7.2-arm64-arm-64bit
requests   : 2.34.2
mlflow     : 3.17.2
pandas     : 3.1.0
jsonschema : 4.26.0


/Users/kmunot/.kiro/crew/scratch/runtime-e9d11948/ipykernel_61810/2204751888.py:19: DeprecationWarning: Accessing jsonschema.__version__ is deprecated and will be removed in a future release. Use importlib.metadata directly to query for jsonschema's version.
  print("jsonschema :", jsonschema.__version__)


In [2]:
PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

load_dotenv(PROJECT_DIR / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY is not set. Put it in the .env file, never in the notebook.")

GEMINI_BASE_URL = os.getenv("GEMINI_BASE_URL", "https://generativelanguage.googleapis.com/v1beta")
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "Google Gemini")
MODEL_NAME = os.getenv("LLM_MODEL", "gemini-3.1-flash-lite")
API_NAME = os.getenv("API_NAME", "Gemini API v1beta, models.generateContent (native REST)")
INPUT_PRICE_PER_MILLION = float(os.getenv("INPUT_PRICE_PER_MILLION"))
OUTPUT_PRICE_PER_MILLION = float(os.getenv("OUTPUT_PRICE_PER_MILLION"))
PRICING_DATE = os.getenv("PRICING_DATE")
CURRENCY = os.getenv("CURRENCY", "USD")
MONTHLY_REQUESTS = int(os.getenv("MONTHLY_REQUESTS", "10000"))
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5001")
EXPERIMENT_NAME = "eduassist_llmops_assignment"
PROMPT_NAME = "eduassist-academic-qa"

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 1024
REQUEST_TIMEOUT_SECONDS = 120

print("API key loaded from environment:", GEMINI_API_KEY is not None)
print("Provider       :", LLM_PROVIDER)
print("Base URL       :", GEMINI_BASE_URL)
print("Model          :", MODEL_NAME)
print("API            :", API_NAME)
print("Input price    :", INPUT_PRICE_PER_MILLION, CURRENCY, "per 1M tokens")
print("Output price   :", OUTPUT_PRICE_PER_MILLION, CURRENCY, "per 1M tokens")
print("Pricing date   :", PRICING_DATE)
print("MLflow URI     :", MLFLOW_TRACKING_URI)

API key loaded from environment: True
Provider       : Google Gemini
Base URL       : https://generativelanguage.googleapis.com/v1beta
Model          : gemini-3.1-flash-lite
API            : Gemini API v1beta, models.generateContent (native REST)
Input price    : 0.25 USD per 1M tokens
Output price   : 1.5 USD per 1M tokens
Pricing date   : 2026-11-03
MLflow URI     : http://127.0.0.1:5001


In [3]:
questions_df = pd.read_csv(PROJECT_DIR / "data" / "questions.csv")
assert len(questions_df) == 5, "Exactly five questions are required"
assert questions_df["question_id"].is_unique
questions_df

,question_id,question
0,Q1,What is the difference between supervised and ...
1,Q2,Why is feature scaling important before traini...
2,Q3,Explain precision and recall with a simple exa...
3,Q4,What is overfitting and how can it be reduced?
4,Q5,Why is experiment tracking important in MLOps?


In [4]:
PROMPT_V1 = """Answer the following student question clearly.
Question: {{question}}"""

PROMPT_V2 = """You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.
Question: {{question}}"""

PROMPT_V3 = """You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}"""

PROMPTS = {
    "v1": {"template": PROMPT_V1, "commit_message": "v1 Basic: baseline prompt that only asks for a clear answer"},
    "v2": {"template": PROMPT_V2, "commit_message": "v2 Structured: add academic-assistant role, concise/accurate instruction and undergraduate-level language"},
    "v3": {"template": PROMPT_V3, "commit_message": "v3 JSON Output: require ONLY valid JSON with answer, confidence (high|medium|low) and key_points; forbid markdown and code fences"},
}

ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {"type": "string", "minLength": 1},
        "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
        "key_points": {"type": "array", "items": {"type": "string"}, "minItems": 1},
    },
    "required": ["answer", "confidence", "key_points"],
    "additionalProperties": False,
}

# fills the {{question}} placeholder in a template
def fill_prompt(template, question):
    return template.replace("{{question}}", question)

print(fill_prompt(PROMPT_V3, "What is overfitting?"))

You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: What is overfitting?


In [5]:
# sends one prompt to the Gemini API and returns timing, status and token usage as a dict
def call_llm(prompt_text, question_id, prompt_version):
    record = {
        "question_id": question_id,
        "prompt_version": prompt_version,
        "model_name": MODEL_NAME,
        "response_model": None,
        "start_time_utc": None,
        "end_time_utc": None,
        "latency_seconds": None,
        "response_status": None,
        "http_status_code": None,
        "finish_reason": None,
        "response_id": None,
        "input_tokens": None,
        "answer_tokens": None,
        "thought_tokens": None,
        "output_tokens": None,
        "total_tokens": None,
        "response_text": None,
        "error_message": "",
    }

    url = GEMINI_BASE_URL + "/models/" + MODEL_NAME + ":generateContent"
    headers = {"Content-Type": "application/json", "X-goog-api-key": GEMINI_API_KEY}
    body = {
        "contents": [{"parts": [{"text": prompt_text}]}],
        "generationConfig": {"temperature": TEMPERATURE, "maxOutputTokens": MAX_OUTPUT_TOKENS},
    }

    start_time = datetime.now(timezone.utc)
    start_counter = time.perf_counter()
    try:
        response = requests.post(url, headers=headers, json=body, timeout=REQUEST_TIMEOUT_SECONDS)
        end_counter = time.perf_counter()
        end_time = datetime.now(timezone.utc)
        record["http_status_code"] = response.status_code

        if response.status_code == 200:
            data = response.json()
            record["response_model"] = data.get("modelVersion")
            record["response_id"] = data.get("responseId")

            usage = data.get("usageMetadata", {})
            record["input_tokens"] = usage.get("promptTokenCount")
            record["answer_tokens"] = usage.get("candidatesTokenCount", 0)
            record["thought_tokens"] = usage.get("thoughtsTokenCount", 0)
            record["total_tokens"] = usage.get("totalTokenCount")
            if record["input_tokens"] is not None:
                record["output_tokens"] = record["answer_tokens"] + record["thought_tokens"]

            candidates = data.get("candidates", [])
            if len(candidates) > 0:
                record["finish_reason"] = candidates[0].get("finishReason")
                text = ""
                for part in candidates[0].get("content", {}).get("parts", []):
                    if not part.get("thought"):
                        text = text + part.get("text", "")
                record["response_text"] = text
                record["response_status"] = "success"
            else:
                record["response_status"] = "error"
                record["error_message"] = "No candidates returned: " + str(data.get("promptFeedback"))[:300]
        else:
            record["response_status"] = "error"
            record["error_message"] = "HTTP " + str(response.status_code) + ": " + response.text[:300]
    except requests.exceptions.RequestException as error:
        end_counter = time.perf_counter()
        end_time = datetime.now(timezone.utc)
        record["response_status"] = "error"
        record["error_message"] = type(error).__name__ + ": " + str(error)[:300]

    record["start_time_utc"] = start_time.isoformat()
    record["end_time_utc"] = end_time.isoformat()
    record["latency_seconds"] = end_counter - start_counter
    return record

In [6]:
# returns input, output and total cost for one request
def calculate_cost(input_tokens, output_tokens):
    input_cost = input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
    output_cost = output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost
    return input_cost, output_cost, total_cost

# checks that a response is valid JSON and matches the required schema
def check_json(response_text):
    if response_text is None:
        return False, "empty response"
    try:
        data = json.loads(response_text)
    except json.JSONDecodeError as error:
        return False, "not valid JSON: " + error.msg + " at character " + str(error.pos)
    try:
        jsonschema.validate(instance=data, schema=ANSWER_SCHEMA)
    except jsonschema.ValidationError as error:
        return False, "schema violation: " + error.message
    return True, ""

print(check_json('{"answer": "A", "confidence": "high", "key_points": ["x"]}'))
print(check_json('```json\n{"answer": "A"}\n```'))
print(check_json('{"answer": "A", "confidence": "sure", "key_points": ["x"]}'))
print(check_json('{"answer": "A", "confidence": "low", "key_points": ["x"], "extra": 1}'))

(True, '')
(False, 'not valid JSON: Expecting value at character 0')
(False, "schema violation: 'sure' is not one of ['high', 'medium', 'low']")
(False, "schema violation: Additional properties are not allowed ('extra' was unexpected)")


In [7]:
smoke = call_llm(fill_prompt(PROMPT_V1, "What is 2 + 2?"), "SMOKE", "v1")
show_keys = ["response_status", "http_status_code", "response_model", "latency_seconds", "input_tokens",
             "answer_tokens", "thought_tokens", "output_tokens", "total_tokens", "finish_reason"]
print({key: smoke[key] for key in show_keys})
print(smoke["response_text"])
assert smoke["response_status"] == "success", smoke["error_message"]
assert smoke["input_tokens"] is not None, "The API response did not include token usage"

{'response_status': 'success', 'http_status_code': 200, 'response_model': 'gemini-3.1-flash-lite', 'latency_seconds': 1.9085524999973131, 'input_tokens': 18, 'answer_tokens': 7, 'thought_tokens': 0, 'output_tokens': 7, 'total_tokens': 25, 'finish_reason': 'STOP'}
2 + 2 = 4


In [8]:
task1_rows = []
for index, row in questions_df.iterrows():
    prompt_text = fill_prompt(PROMPT_V1, row["question"])
    result = call_llm(prompt_text, row["question_id"], "v1")
    result["question"] = row["question"]
    task1_rows.append(result)
    print(row["question_id"], result["response_status"], round(result["latency_seconds"], 3), "s")

task1_df = pd.DataFrame(task1_rows)
assert (task1_df["response_status"] == "success").all(), "A request failed - check error_message and re-run"
latency_table = task1_df[["question_id", "prompt_version", "model_name", "start_time_utc", "end_time_utc", "latency_seconds", "response_status"]]
latency_table

Q1 success 6.715 s


Q2 success 11.332 s


Q3 success 4.64 s


Q4 success 20.401 s


Q5 success 2.97 s


,question_id,prompt_version,model_name,start_time_utc,end_time_utc,latency_seconds,response_status
0,Q1,v1,gemini-3.1-flash-lite,2026-11-03T11:22:39.409604+00:00,2026-11-03T11:22:46.124808+00:00,6.715202,success
1,Q2,v1,gemini-3.1-flash-lite,2026-11-03T11:22:46.128239+00:00,2026-11-03T11:22:57.460210+00:00,11.331981,success
2,Q3,v1,gemini-3.1-flash-lite,2026-11-03T11:22:57.463188+00:00,2026-11-03T11:23:02.102769+00:00,4.639575,success
3,Q4,v1,gemini-3.1-flash-lite,2026-11-03T11:23:02.104462+00:00,2026-11-03T11:23:22.505490+00:00,20.401055,success
4,Q5,v1,gemini-3.1-flash-lite,2026-11-03T11:23:22.508209+00:00,2026-11-03T11:23:25.478083+00:00,2.969868,success


In [9]:
latencies = task1_df["latency_seconds"]
latency_summary = pd.DataFrame({
    "statistic": ["mean", "median", "minimum", "maximum", "standard deviation (sample, ddof=1)"],
    "latency_seconds": [latencies.mean(), latencies.median(), latencies.min(), latencies.max(), latencies.std(ddof=1)],
})
latency_summary.to_csv(OUTPUT_DIR / "latency_summary.csv", index=False)

slowest = task1_df.loc[latencies.idxmax()]
fastest = task1_df.loc[latencies.idxmin()]
print("Slowest:", slowest["question_id"], round(slowest["latency_seconds"], 3), "s,",
      slowest["output_tokens"], "output tokens (thinking:", slowest["thought_tokens"], ")")
print("Fastest:", fastest["question_id"], round(fastest["latency_seconds"], 3), "s,",
      fastest["output_tokens"], "output tokens (thinking:", fastest["thought_tokens"], ")")
print("Correlation between output tokens and latency:", round(task1_df["output_tokens"].corr(task1_df["latency_seconds"]), 3))
print("Mean seconds per output token:", round((task1_df["latency_seconds"] / task1_df["output_tokens"]).mean(), 5))
latency_summary

Slowest: Q4 20.401 s, 729 output tokens (thinking: 0 )
Fastest: Q5 2.97 s, 640 output tokens (thinking: 0 )
Correlation between output tokens and latency: 0.93
Mean seconds per output token: 0.01333


,statistic,latency_seconds
0,mean,9.211536
1,median,6.715202
2,minimum,2.969868
3,maximum,20.401055
4,"standard deviation (sample, ddof=1)",6.996330


### Task 1 interpretation

**Interpretation.** Mean latency was 9.21 s (median 6.72 s, range 2.97–20.40 s, sample SD 7.00 s, n−1 convention).
The slowest request was **Q4** at 20.40 s. It also produced the most output (729 tokens versus 640 for the fastest request, Q5 at 2.97 s),
and across the five requests latency correlated strongly with output length (r = 0.93, indicative only with n = 5).
Output length alone does not explain the spread, though. The same five prompts produced identical token counts
(632, 703, 638, 729, 640) in the v1 evaluation later, but there the latencies were only 2.46–9.46 s (mean 4.12 s), Q4 took 2.91 s,
and the slowest request was Q5. So most of Q4's 20.4 s was a transient delay rather than generation time. The Gemini API
does not report server-side processing time, so network jitter, provider queueing and load at the time of the call cannot be
separated here; these are possible reasons, not measured ones. The smoke-test call opened the connection beforehand,
so one-time connection setup is unlikely to explain it.

## Task 2: Tokens and cost

$$\text{Input Cost} = \frac{\text{Input Tokens}}{1{,}000{,}000} \times \text{Input Price per 1M}$$

$$\text{Output Cost} = \frac{\text{Output Tokens}}{1{,}000{,}000} \times \text{Output Price per 1M}$$

$$\text{Total Cost per Request} = \text{Input Cost} + \text{Output Cost}$$

$$\text{Projected Monthly Cost} = \text{Average Cost per Request} \times 10{,}000$$

Gemini bills thinking tokens as output tokens, so **Output Tokens = answer tokens + thinking tokens** (`candidatesTokenCount + thoughtsTokenCount`).

In [10]:
input_costs = []
output_costs = []
total_costs = []
for index, row in task1_df.iterrows():
    input_cost, output_cost, total_cost = calculate_cost(row["input_tokens"], row["output_tokens"])
    input_costs.append(input_cost)
    output_costs.append(output_cost)
    total_costs.append(total_cost)

task1_df["input_cost"] = input_costs
task1_df["output_cost"] = output_costs
task1_df["total_cost"] = total_costs
task1_df["currency"] = CURRENCY
task1_df["input_price_per_million"] = INPUT_PRICE_PER_MILLION
task1_df["output_price_per_million"] = OUTPUT_PRICE_PER_MILLION
task1_df["pricing_date"] = PRICING_DATE

assert (task1_df["total_tokens"] == task1_df["input_tokens"] + task1_df["output_tokens"]).all(), "API total_tokens does not equal input + answer + thinking tokens"

average_cost_per_request = task1_df["total_cost"].mean()
projected_monthly_cost = average_cost_per_request * MONTHLY_REQUESTS

cost_table = task1_df[["question_id", "input_tokens", "answer_tokens", "thought_tokens", "output_tokens", "total_tokens", "input_cost", "output_cost", "total_cost"]]
print("Average cost per request:", f"{average_cost_per_request:.8f}", CURRENCY)
print("Projected monthly cost for", MONTHLY_REQUESTS, "requests:", f"{projected_monthly_cost:.4f}", CURRENCY)
print("Share of cost from output tokens:", f"{task1_df['output_cost'].sum() / task1_df['total_cost'].sum():.1%}")

csv_columns = ["question_id", "question", "prompt_version", "model_name", "response_model",
               "start_time_utc", "end_time_utc", "latency_seconds", "response_status", "http_status_code",
               "finish_reason", "response_id", "input_tokens", "answer_tokens", "thought_tokens", "output_tokens",
               "total_tokens", "input_price_per_million", "output_price_per_million", "input_cost", "output_cost",
               "total_cost", "currency", "pricing_date", "error_message"]
task1_df[csv_columns].to_csv(OUTPUT_DIR / "latency_tokens_cost.csv", index=False)
cost_table.style.format({"input_cost": "{:.8f}", "output_cost": "{:.8f}", "total_cost": "{:.8f}"})

Average cost per request: 0.00100765 USD
Projected monthly cost for 10000 requests: 10.0765 USD
Share of cost from output tokens: 99.5%


,question_id,input_tokens,answer_tokens,thought_tokens,output_tokens,total_tokens,input_cost,output_cost,total_cost
0,Q1,20,632,0,632,652,0.00000500,0.00094800,0.00095300
1,Q2,22,703,0,703,725,0.00000550,0.00105450,0.00106000
2,Q3,19,638,0,638,657,0.00000475,0.00095700,0.00096175
3,Q4,20,729,0,729,749,0.00000500,0.00109350,0.00109850
4,Q5,20,640,0,640,660,0.00000500,0.00096000,0.00096500


**Worked example** (Q2 row above): 22 input tokens and 703 output tokens → input cost 22/1,000,000 × 0.25 = $0.0000055;
output cost 703/1,000,000 × 1.50 = $0.0010545; total $0.0010600.

### Cost-optimisation recommendation

**Recommendation: cap and shape the response length.** Output tokens made up 99.5% of the Task 1 cost (the output price of
$1.50/1M is 6× the input price of $0.25/1M), and v1 averaged 668.4 output tokens (no thinking tokens were reported). Asking for a
concise answer, as v2 and v3 do, and setting a lower `maxOutputTokens` for short academic questions would cut output tokens.
At 668.4 → about 250 output tokens, the average cost per request falls from $0.0010077 to about $0.00038, i.e. $10.08 → $3.80 per
10,000 requests. The measured v3 prompt (176 output tokens on average, $0.000285, $2.85 per 10,000 requests) shows the same effect.
Trade-off: answers that are too short may drop explanation students need, and a hard cap can truncate a JSON object (making it
invalid), so any cap must be validated against `finishReason` and `valid_json`.

In [11]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print("Experiment:", experiment.name, "| id:", experiment.experiment_id)

# returns the registered version that already has this exact template, or None
def find_existing_version(template):
    version_number = 1
    while True:
        existing = mlflow.genai.load_prompt(PROMPT_NAME, version=version_number, allow_missing=True)
        if existing is None:
            return None
        if existing.template == template:
            return existing
        version_number = version_number + 1

registered = {}
for label in ["v1", "v2", "v3"]:
    template = PROMPTS[label]["template"]
    prompt_version = find_existing_version(template)
    if prompt_version is None:
        prompt_version = mlflow.genai.register_prompt(
            name=PROMPT_NAME,
            template=template,
            commit_message=PROMPTS[label]["commit_message"],
            tags={"prompt_label": label, "task": "eduassist-academic-qa"},
        )
        print("Registered", label, "as version", prompt_version.version)
    else:
        print("Reusing", label, "-> already registered as version", prompt_version.version)
    registered[label] = prompt_version

pd.DataFrame([
    {"label": label, "registry_version": pv.version, "uri": pv.uri, "commit_message": pv.commit_message}
    for label, pv in registered.items()
])

2026/11/03 16:53:25 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


Experiment: eduassist_llmops_assignment | id: 1


2026/11/03 16:53:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist-academic-qa, version 1


Registered v1 as version 1


2026/11/03 16:53:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist-academic-qa, version 2


Registered v2 as version 2


2026/11/03 16:53:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist-academic-qa, version 3


Registered v3 as version 3


,label,registry_version,uri,commit_message
0,v1,1,prompts:/eduassist-academic-qa/1,v1 Basic: baseline prompt that only asks for a...
1,v2,2,prompts:/eduassist-academic-qa/2,"v2 Structured: add academic-assistant role, co..."
2,v3,3,prompts:/eduassist-academic-qa/3,v3 JSON Output: require ONLY valid JSON with a...


In [12]:
evaluation_rows = []
run_ids = {}

for label in ["v1", "v2", "v3"]:
    registry_version = registered[label].version
    with mlflow.start_run(run_name="prompt_" + label) as run:
        run_ids[label] = run.info.run_id
        prompt = mlflow.genai.load_prompt(PROMPT_NAME, version=registry_version)

        mlflow.log_params({
            "prompt_label": label,
            "prompt_name": PROMPT_NAME,
            "prompt_registry_version": registry_version,
            "prompt_uri": prompt.uri,
            "provider": LLM_PROVIDER,
            "model_name": MODEL_NAME,
            "api": API_NAME,
            "temperature": TEMPERATURE,
            "max_output_tokens": MAX_OUTPUT_TOKENS,
            "input_price_per_million": INPUT_PRICE_PER_MILLION,
            "output_price_per_million": OUTPUT_PRICE_PER_MILLION,
            "pricing_date": PRICING_DATE,
            "currency": CURRENCY,
            "num_questions": len(questions_df),
        })
        mlflow.set_tags({"assignment": "MLOps Assignment 2", "prompt_label": label})

        version_rows = []
        for index, row in questions_df.iterrows():
            prompt_text = prompt.format(question=row["question"])
            with mlflow.start_span(name=label + "_" + row["question_id"], span_type="LLM") as span:
                span.set_inputs({"question_id": row["question_id"], "prompt": prompt_text})
                result = call_llm(prompt_text, row["question_id"], label)
                span.set_outputs({"response_text": result["response_text"], "status": result["response_status"]})
                trace_id = span.trace_id

            is_valid, validation_error = check_json(result["response_text"])
            input_cost, output_cost, total_cost = calculate_cost(result["input_tokens"] or 0, result["output_tokens"] or 0)

            result["question"] = row["question"]
            result["registry_prompt_name"] = PROMPT_NAME
            result["registry_prompt_version"] = registry_version
            result["input_cost"] = input_cost
            result["output_cost"] = output_cost
            result["total_cost"] = total_cost
            result["currency"] = CURRENCY
            result["is_valid_json"] = is_valid
            result["validation_error"] = validation_error
            result["trace_id"] = trace_id
            result["mlflow_run_id"] = run.info.run_id
            version_rows.append(result)
            print(label, row["question_id"], result["response_status"],
                  round(result["latency_seconds"], 3), "s | valid_json =", is_valid)

        mlflow.flush_trace_async_logging()
        for result in version_rows:
            mlflow.log_feedback(
                trace_id=result["trace_id"],
                name="valid_json",
                value=bool(result["is_valid_json"]),
                rationale=result["validation_error"] or "Parsed with json.loads and matched the required schema",
            )

        version_df = pd.DataFrame(version_rows)
        assert len(version_df) == 5
        assert (version_df["response_status"] == "success").all(), "A request failed in " + label

        metrics = {
            "avg_latency_seconds": version_df["latency_seconds"].mean(),
            "avg_input_tokens": version_df["input_tokens"].mean(),
            "avg_output_tokens": version_df["output_tokens"].mean(),
            "avg_cost_per_request": version_df["total_cost"].mean(),
            "valid_json": version_df["is_valid_json"].sum() / len(version_df),
            "avg_thought_tokens": version_df["thought_tokens"].mean(),
            "median_latency_seconds": version_df["latency_seconds"].median(),
            "max_latency_seconds": version_df["latency_seconds"].max(),
            "std_latency_seconds": version_df["latency_seconds"].std(ddof=1),
            "avg_total_tokens": version_df["total_tokens"].mean(),
            "total_cost_5_requests": version_df["total_cost"].sum(),
            "projected_monthly_cost": version_df["total_cost"].mean() * MONTHLY_REQUESTS,
            "success_rate": (version_df["response_status"] == "success").mean(),
        }
        mlflow.log_metrics(metrics)

        per_question_path = OUTPUT_DIR / ("per_question_" + label + ".csv")
        version_df.to_csv(per_question_path, index=False)
        mlflow.log_artifact(str(per_question_path), artifact_path="evaluation")
        mlflow.log_text(prompt.template, "prompt/prompt_template.txt")
        evaluation_rows.extend(version_rows)

eval_df = pd.DataFrame(evaluation_rows)
eval_columns = ["prompt_version", "registry_prompt_name", "registry_prompt_version", "question_id", "question",
                "model_name", "response_model", "start_time_utc", "end_time_utc", "response_status",
                "http_status_code", "finish_reason", "latency_seconds", "input_tokens", "answer_tokens",
                "thought_tokens", "output_tokens", "total_tokens", "input_cost", "output_cost", "total_cost",
                "currency", "is_valid_json", "validation_error", "mlflow_run_id", "trace_id", "response_text"]
eval_df[eval_columns].to_csv(OUTPUT_DIR / "prompt_evaluation_results.csv", index=False)
eval_df[["prompt_version", "question_id", "latency_seconds", "input_tokens", "output_tokens", "total_cost", "is_valid_json", "validation_error"]]

v1 Q1 success 2.46 s | valid_json = False


v1 Q2 success 3.151 s | valid_json = False


v1 Q3 success 2.619 s | valid_json = False


v1 Q4 success 2.909 s | valid_json = False


v1 Q5 success 9.463 s | valid_json = False


🏃 View run prompt_v1 at: http://127.0.0.1:5001/#/experiments/1/runs/e5ce614647464a48a7f268e8cd3a3606
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


... (10 outputs left out)

v3 Q5 success 1.274 s | valid_json = True


🏃 View run prompt_v3 at: http://127.0.0.1:5001/#/experiments/1/runs/6082194924ea47f9b03e0d450984f2fa
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


,prompt_version,question_id,latency_seconds,input_tokens,output_tokens,total_cost,is_valid_json,validation_error
0,v1,Q1,2.460097,20,632,0.000953,False,not valid JSON: Expecting value at character 0
1,v1,Q2,3.150505,22,703,0.001060,False,not valid JSON: Expecting value at character 0
2,v1,Q3,2.618635,19,638,0.000962,False,not valid JSON: Expecting value at character 0
3,v1,Q4,2.909328,20,729,0.001099,False,not valid JSON: Expecting value at character 0
4,v1,Q5,9.462626,20,640,0.000965,False,not valid JSON: Expecting value at character 0
5,v2,Q1,1.852048,44,414,0.000632,False,not valid JSON: Expecting value at character 0
6,v2,Q2,2.550400,46,462,0.000705,False,not valid JSON: Expecting value at character 0
7,v2,Q3,3.946881,43,453,0.000690,False,not valid JSON: Expecting value at character 0
8,v2,Q4,2.375831,44,602,0.000914,False,not valid JSON: Expecting value at character 0
9,v2,Q5,6.439869,44,396,0.000605,False,not valid JSON: Expecting value at character 0


[Trace(trace_id='tr-6e3acf294955c1ebb2fb2e9dc0a14e5c'), Trace(trace_id='tr-ee809b2e9aa1be688fff0105ea84f32d'), Trace(trace_id='tr-6ed7314c8b9def49564c5114d7b914ed'), Trace(trace_id='tr-e4e54570fe5c3697ca84de4878104c81'), Trace(trace_id='tr-dfd496b36e0b5a195728f384c1914a85'), Trace(trace_id='tr-083833388133c61723db527e2411306e'), Trace(trace_id='tr-280369c9faed5e89a6b26b493ec6c50e'), Trace(trace_id='tr-b91d39931f52cd9e9ebe9b4b9853e444'), Trace(trace_id='tr-59e2e5028fe72897f44f9bbe242eccbd'), Trace(trace_id='tr-4d05e7065966d30dcbbe0f1bbdfd4d8a')]

## Task 3 results: comparison and promotion

**Definition of `valid_json`.** A response counts as valid only if (1) the whole response text parses with `json.loads`
(no stripping of code fences, no repair), (2) the top-level value is an object with exactly the keys `answer`, `confidence`,
`key_points`, and (3) `answer` is a non-empty string, `confidence` is `high`, `medium` or `low`, and `key_points` is a non-empty
list of strings. `valid_json = valid responses / total responses`, computed with the same validator for every version.
v1 and v2 never ask for JSON, so a score of 0.0 for them is expected; the metric shows that only v3 meets the downstream JSON contract.

**Selection rule.** (1) The JSON contract is a hard requirement, so keep only versions with the highest `valid_json`.
(2) Among those prefer lower average cost, then lower average latency. (3) Quality is checked by the manual review below.

In [13]:
comparison_rows = []
for label in ["v1", "v2", "v3"]:
    rows = eval_df[eval_df["prompt_version"] == label]
    comparison_rows.append({
        "Prompt Version": label,
        "Registry Version": registered[label].version,
        "Avg Latency": rows["latency_seconds"].mean(),
        "Avg Input Tokens": rows["input_tokens"].mean(),
        "Avg Output Tokens": rows["output_tokens"].mean(),
        "Avg Cost": rows["total_cost"].mean(),
        "Valid JSON Rate": rows["is_valid_json"].sum() / len(rows),
        "Projected Monthly Cost": rows["total_cost"].mean() * MONTHLY_REQUESTS,
    })
comparison_df = pd.DataFrame(comparison_rows)

best_json_rate = comparison_df["Valid JSON Rate"].max()
candidates = comparison_df[comparison_df["Valid JSON Rate"] == best_json_rate]
candidates = candidates.sort_values(["Avg Cost", "Avg Latency"])
selected_label = candidates.iloc[0]["Prompt Version"]

recommended = []
for label in comparison_df["Prompt Version"]:
    if label == selected_label:
        recommended.append("Yes")
    else:
        recommended.append("No")
comparison_df["Recommended?"] = recommended
comparison_df.to_csv(OUTPUT_DIR / "prompt_comparison.csv", index=False)
print("Selected for production:", selected_label, "(registry version", registered[selected_label].version, ")")
comparison_df

Selected for production: v3 (registry version 3 )


,Prompt Version,Registry Version,Avg Latency,Avg Input Tokens,Avg Output Tokens,Avg Cost,Valid JSON Rate,Projected Monthly Cost,Recommended?
0,v1,1,4.120238,20.2,668.4,0.001008,0.0,10.0765,No
1,v2,2,3.433006,44.2,465.4,0.000709,0.0,7.0915,No
2,v3,3,2.574678,85.2,176.0,0.000285,1.0,2.8530,Yes


### Manual quality review

I read all 15 responses in `outputs/prompt_evaluation_results.csv` (column `response_text`). Scores: 1 = poor, 5 = excellent, the rounded mean of
correctness (no factual errors), clarity for an undergraduate, and completeness (answers every part of the question, e.g. Q3 "with a simple example"
and Q4 "how can it be reduced").

| Version | Q1 | Q2 | Q3 | Q4 | Q5 | Avg | Notes |
|---|---|---|---|---|---|---|---|
| v1 | 4 | 4 | 5 | 5 | 4 | 4.4 | Accurate and complete, but long, heavy markdown and headings; Q3 uses a cat-photo example |
| v2 | 5 | 5 | 5 | 5 | 4 | 4.8 | Accurate, concise, undergraduate level; Q3 uses a worked spam-email example |
| v3 | 4 | 4 | 4 | 4 | 4 | 4.0 | Correct and parseable; short answers with a useful `key_points` list; less depth, but Q3 still includes a spam-filter example |

In [14]:
selected_version = registered[selected_label].version
mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="production", version=selected_version)
production_prompt = mlflow.genai.load_prompt("prompts:/" + PROMPT_NAME + "@production")
print("production ->", production_prompt.name, "version", production_prompt.version)
assert production_prompt.version == selected_version
with mlflow.start_run(run_id=run_ids[selected_label]):
    mlflow.set_tag("promoted_alias", "production")

production -> eduassist-academic-qa version 3


🏃 View run prompt_v3 at: http://127.0.0.1:5001/#/experiments/1/runs/6082194924ea47f9b03e0d450984f2fa
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


### Promotion justification

**Promotion decision: v3 (registry version 3) → alias `production`.**

*JSON validity.* EduAssist's downstream application expects strict JSON with `answer`, `confidence` and `key_points`. v3 achieved a
valid_json rate of 1.0 (5/5 responses parsed and matched the schema), while v1 and v2 scored 0.0 because they return free-form markdown
prose that cannot be parsed. A prompt that breaks the output contract would need a fragile repair step, so v1 and v2 are not
production-ready for this use case.

*Tokens and cost.* v3 uses more input tokens (85.2 versus 20.2 for v1 and 44.2 for v2) because it embeds the schema, but far fewer
output tokens (176.0 versus 668.4 and 465.4). Output tokens cost six times as much as input tokens, so v3's average cost per request is
$0.000285 versus $0.001008 (v1) and $0.000709 (v2): a projected $2.85 versus $10.08 and $7.09 per 10,000 requests.

*Latency.* Average latency was 2.57 s for v3 versus 4.12 s for v1 and 3.43 s for v2. Latency was noisy (the same v1 prompts took
2.5–20 s across runs), so the ranking is indicative; shorter generation plausibly helps.

*Quality.* In my manual review v3 averaged 4.0/5 against 4.4 for v1 and 4.8 for v2. Its answers were correct and still included an
example where asked (Q3), but they are shorter and less detailed than v2. That trade-off is accepted because v3 is the only version that
meets the JSON contract, and its `key_points` list gives the UI structure it can render.

*Risk.* Five questions is a small sample; the alias can be moved back to another version if production monitoring shows invalid JSON or weak answers.

In [15]:
metadata = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version.split()[0],
    "requests_version": requests.__version__,
    "mlflow_version": mlflow.__version__,
    "provider": LLM_PROVIDER,
    "model_name": MODEL_NAME,
    "response_models": sorted(eval_df["response_model"].dropna().unique().tolist()),
    "api": API_NAME,
    "input_price_per_million": INPUT_PRICE_PER_MILLION,
    "output_price_per_million": OUTPUT_PRICE_PER_MILLION,
    "pricing_date": PRICING_DATE,
    "currency": CURRENCY,
    "monthly_requests": MONTHLY_REQUESTS,
    "temperature": TEMPERATURE,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
    "experiment_name": EXPERIMENT_NAME,
    "experiment_id": experiment.experiment_id,
    "prompt_name": PROMPT_NAME,
    "prompt_registry_versions": {label: pv.version for label, pv in registered.items()},
    "mlflow_run_ids": run_ids,
    "production_alias_version": selected_version,
    "production_prompt_label": selected_label,
}
with open(OUTPUT_DIR / "run_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)
print(json.dumps(metadata, indent=2))

{
  "generated_at_utc": "2026-11-03T11:24:24.389992+00:00",
  "python_version": "3.12.13",
  "requests_version": "2.34.2",
  "mlflow_version": "3.17.2",
  "provider": "Google Gemini",
  "model_name": "gemini-3.1-flash-lite",
  "response_models": [
    "gemini-3.1-flash-lite"
  ],
  "api": "Gemini API v1beta, models.generateContent (native REST)",
  "input_price_per_million": 0.25,
  "output_price_per_million": 1.5,
  "pricing_date": "2026-11-03",
  "currency": "USD",
  "monthly_requests": 10000,
  "temperature": 0.0,
  "max_output_tokens": 1024,
  "experiment_name": "eduassist_llmops_assignment",
  "experiment_id": "1",
  "prompt_name": "eduassist-academic-qa",
  "prompt_registry_versions": {
    "v1": 1,
    "v2": 2,
    "v3": 3
  },
  "mlflow_run_ids": {
    "v1": "e5ce614647464a48a7f268e8cd3a3606",
    "v2": "2085e35041324735b80d81ebc7c6d287",
    "v3": "6082194924ea47f9b03e0d450984f2fa"
  },
  "production_alias_version": 3,
  "production_prompt_label": "v3"
}


In [16]:
assert len(task1_df) == 5
assert set(eval_df["prompt_version"]) == {"v1", "v2", "v3"}
for label in ["v1", "v2", "v3"]:
    assert eval_df[eval_df["prompt_version"] == label]["question_id"].tolist() == questions_df["question_id"].tolist()
assert (eval_df["response_status"] == "success").all()
assert eval_df["input_tokens"].notna().all() and eval_df["output_tokens"].notna().all()
assert (task1_df["latency_seconds"] >= 0).all() and (eval_df["latency_seconds"] >= 0).all()
assert (eval_df["total_cost"] >= 0).all()
assert abs(projected_monthly_cost - task1_df["total_cost"].mean() * 10000) < 1e-12
assert comparison_df["Valid JSON Rate"].between(0, 1).all()
assert (comparison_df["Recommended?"] == "Yes").sum() == 1

finished_runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
for label in ["v1", "v2", "v3"]:
    run_row = finished_runs[finished_runs["run_id"] == run_ids[label]].iloc[0]
    for metric_name in ["avg_latency_seconds", "avg_input_tokens", "avg_output_tokens", "avg_cost_per_request", "valid_json"]:
        assert pd.notna(run_row["metrics." + metric_name]), label + " is missing " + metric_name
assert mlflow.genai.load_prompt("prompts:/" + PROMPT_NAME + "@production").version == selected_version

for file_name in ["latency_tokens_cost.csv", "prompt_evaluation_results.csv"]:
    assert (OUTPUT_DIR / file_name).exists()

key_text = GEMINI_API_KEY
for path in OUTPUT_DIR.iterdir():
    if path.is_file():
        assert key_text not in path.read_text(errors="ignore"), "API key found in " + path.name
print("All checks passed.")

All checks passed.


**Reproduce:** see README.md (Podman MLflow server, venv, `.env`, Restart & Run All).

**Limitations:** five questions is a small sample (latency statistics and correlations are indicative only);
latency depends on network and provider load at run time; LLM output is non-deterministic even at temperature 0,
so a re-run gives slightly different tokens, costs and possibly JSON validity; prices are the published standard
rates on 2026-11-03 and exclude taxes; Gemini bills thinking tokens as output, so token counts include them;
`valid_json` checks structure, not correctness, which is why a manual quality review was added.